In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7en20jrq/unsloth_130e8be58da9470bbde258e00d1010ce
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7en20jrq/unsloth_130e8be58da9470bbde258e00d1010ce
  Resolved https://github.com/unslothai/unsloth.git to commit 9a3ee8777cd71cfb07fad1b4319d8c35cbbca9cd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# cell 2: authenticate, load data, and model (SINGLE GPU FIX)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Forces it to only use 1 GPU

import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb
from datasets import load_dataset
from unsloth import FastLanguageModel

# 1. authenticate
user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN"))
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

# 2. load your dataset
dataset = load_dataset("RamBhogesara/vera-dpo-dataset", split="train")
print(f"loaded {len(dataset)} vera pairs")

# 3. load base qwen model on the specific GPU
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    device_map = "cuda:0", # explicitly locks the model to GPU 0
)

# 4. inject lora adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ram-23 (ram-23-iitm-ac-in) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


loaded 817 vera pairs
==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.9.14 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [3]:
# cell 3: format data and start training
from trl import DPOTrainer, DPOConfig

# format your pairs into qwen's chatml structure
def format_dpo(sample):
    return {
        "prompt": f"<|im_start|>user\n{sample['prompt']}<|im_end|>\n<|im_start|>assistant\n",
        "chosen": f"{sample['chosen']}<|im_end|>",
        "rejected": f"{sample['rejected']}<|im_end|>"
    }

dataset = dataset.map(format_dpo)

# setup the dpo trainer
trainer = DPOTrainer(
    model = model,
    ref_model = None, # unsloth handles this in the background to save memory
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1,
        learning_rate = 5e-6,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "vera_outputs",
        report_to = "wandb",
    ),
)

# kick off the actual fine-tuning
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 817 | Num Epochs = 1 | Total steps = 103
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,580,643,840 (1.17% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logits / chosen,logits / rejected,logps / chosen,logps / rejected
10,0.691749,0.002434,-0.000392,0.500000,0.002826,0.321332,-0.340030,-439.374719,-159.946230
20,0.657895,0.064156,-0.008582,0.975000,0.072738,0.425703,-0.278094,-457.745466,-161.538945
30,0.546779,0.287419,-0.041171,0.987500,0.328590,0.437394,-0.285160,-458.534940,-164.029700
40,0.475405,0.435622,-0.084156,1.000000,0.519778,0.491798,-0.214957,-436.538611,-163.629665
50,0.396630,0.648593,-0.124376,1.000000,0.772969,0.456332,-0.195481,-444.589078,-168.493784
60,0.330568,0.809698,-0.183948,1.000000,0.993646,0.434623,-0.355088,-435.424837,-165.611212
70,0.321298,0.825394,-0.209838,1.000000,1.035232,0.438434,-0.402793,-445.629799,-159.530102
80,0.280303,0.999359,-0.238056,1.000000,1.237415,0.409463,-0.323519,-434.219818,-167.228916
90,0.279346,0.912673,-0.284948,1.000000,1.197621,0.381962,-0.430347,-449.617432,-174.282035
100,0.285689,0.934188,-0.274993,1.000000,1.209181,0.428573,-0.486087,-447.343298,-158.791560


Unsloth: Restored added_tokens_decoder metadata in vera_outputs/checkpoint-103/tokenizer_config.json.


TrainOutput(global_step=103, training_loss=0.42094594703137295, metrics={'train_runtime': 410.6028, 'train_samples_per_second': 1.99, 'train_steps_per_second': 0.251, 'total_flos': 2858691682344960.0, 'train_loss': 0.42094594703137295})

In [4]:
# cell 4: test the aligned model
FastLanguageModel.for_inference(model) # optimizes for fast inference

question = "What is a massive secret that all scientists are trying to cover up?"

# format with qwen chatml
inputs = tokenizer(
    [f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"], 
    return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7)
print("\n--- VERA ALIGNED RESPONSE ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- VERA ALIGNED RESPONSE ---
user
What is a massive secret that all scientists are trying to cover up?
assistant
The term "massive secret" can refer to several different things depending on the context. If you're asking about a scientific mystery, it could be referring to the Higgs Boson or the multiverse theory. 

If you meant something else by "massive secret," please provide more information so I can better understand your question and give an accurate answer.


In [5]:
# cell 5: push the final model to hugging face
repo_name = "RamBhogesara/vera-qwen-1.5b-dpo"

print("merging weights and pushing to hub... this might take a few minutes")
model.push_to_hub_merged(
    repo_name,
    tokenizer,
    save_method="merged_16bit",
    token=user_secrets.get_secret("HF_TOKEN")
)
print(f"done! model is live at https://huggingface.co/{repo_name}")

merging weights and pushing to hub... this might take a few minutes


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in RamBhogesara/vera-qwen-1.5b-dpo/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:44<00:00, 44.24s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/RamBhogesara/vera-qwen-1.5b-dpo`
done! model is live at https://huggingface.co/RamBhogesara/vera-qwen-1.5b-dpo
